# L2 - LLM Missing Values & Outlier Analysis
Analisi sistematica di valori mancanti e outlier su tutti i CSV Olist.
Per ogni tabella: classificazione MCAR/MAR/MNAR + detection outlier IQR/Z-score + narrativa LLM.

In [ ]:
import os, json, re, warnings
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')

_HERE      = Path(os.path.abspath('__file__')).parent
_ROOT      = _HERE.parent
RAW_PATH   = str(_ROOT / '1_raw_data') + os.sep
OUT_PATH   = str(_HERE / 'outputs') + os.sep
os.makedirs(OUT_PATH, exist_ok=True)
print('RAW_PATH:', RAW_PATH)
print('OUT_PATH:', OUT_PATH)

In [ ]:
import requests

LLM_PROVIDER      = 'ollama'
OLLAMA_MODEL      = 'llama3.2:3b'
OLLAMA_URL        = 'http://localhost:11434/api/chat'
OPENAI_MODEL      = 'gpt-4o-mini'
ANTHROPIC_MODEL   = 'claude-haiku-4-5-20251001'
OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY', '')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

def call_llm(system_prompt, user_prompt, max_tokens=1200):
    try:
        if LLM_PROVIDER == 'ollama':
            r = requests.post(OLLAMA_URL, json={
                'model': OLLAMA_MODEL, 'stream': False,
                'messages': [{'role':'system','content':system_prompt},
                             {'role':'user','content':user_prompt}]
            }, timeout=120)
            r.raise_for_status()
            return r.json()['message']['content']
        elif LLM_PROVIDER == 'openai':
            import openai
            client = openai.OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(model=OPENAI_MODEL,
                messages=[{'role':'system','content':system_prompt},
                           {'role':'user','content':user_prompt}])
            return resp.choices[0].message.content
        elif LLM_PROVIDER == 'anthropic':
            import anthropic
            client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            msg = client.messages.create(model=ANTHROPIC_MODEL, max_tokens=max_tokens,
                system=system_prompt,
                messages=[{'role':'user','content':user_prompt}])
            return msg.content[0].text
    except Exception as e:
        return f'[LLM ERROR] {e}'

print(f'Provider: {LLM_PROVIDER}')

In [ ]:
TABLES = {
    'orders':      'olist_orders_dataset.csv',
    'customers':   'olist_customers_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'products':    'olist_products_dataset.csv',
    'sellers':     'olist_sellers_dataset.csv',
    'payments':    'olist_order_payments_dataset.csv',
    'reviews':     'olist_order_reviews_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
}

datasets = {}
for name, fname in TABLES.items():
    fpath = RAW_PATH + fname
    if os.path.exists(fpath):
        datasets[name] = pd.read_csv(fpath)
        print(f'  [{name}] {len(datasets[name]):,} righe x {datasets[name].shape[1]} colonne')
    else:
        print(f'  [SKIP] {fname} non trovato')

In [ ]:
SYSTEM_MISSING = (
    'Sei un senior Data Engineer specializzato in e-commerce Data Warehouses. '
    'Stai analizzando il dataset Olist, marketplace brasiliano con ~100.000 ordini (2016-2018). '
    'Classifica i pattern di valori mancanti come MCAR/MAR/MNAR con motivazione domain-aware. '
    'Suggerisci la strategia ottimale di gestione (flag, imputazione, esclusione). '
    'Scrivi in italiano. Sii preciso e orientato al business.'
)

SYSTEM_OUTLIER = (
    'Sei un senior Data Analyst per Olist, marketplace brasiliano. '
    'Interpreti risultati di outlier detection e produci narrative business. '
    'Per ogni outlier: cause plausibili, impatto sul DW, strategia raccomandata (flag, cap, rimozione). '
    'Scrivi in italiano.'
)

print('Prompt caricati.')

In [ ]:
def classify_missingness(df):
    result = {}
    for col in df.columns:
        null_pct = df[col].isna().mean() * 100
        if null_pct == 0:
            continue
        if null_pct < 0.5:
            kind = 'MCAR'
        elif df.select_dtypes(include='object').shape[1] > 0:
            corr_found = False
            for cat_col in df.select_dtypes(include='object').columns:
                if cat_col == col:
                    continue
                grp_std = df.groupby(cat_col)[col].apply(lambda x: x.isna().mean()).std()
                if grp_std > 0.05:
                    corr_found = True
                    break
            kind = 'MAR' if corr_found else 'MNAR'
        else:
            kind = 'MNAR'
        result[col] = {'pct': round(null_pct, 2), 'type': kind}
    return result


def detect_outliers_iqr(df, threshold=1.5):
    results = []
    for col in df.select_dtypes(include='number').columns:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR == 0:
            continue
        lb, ub = Q1 - threshold * IQR, Q3 + threshold * IQR
        n_out = ((df[col] < lb) | (df[col] > ub)).sum()
        if n_out > 0:
            results.append({
                'column': col, 'method': 'IQR',
                'n_outliers': int(n_out),
                'pct': round(n_out / len(df) * 100, 2),
                'lower_bound': round(float(lb), 2),
                'upper_bound': round(float(ub), 2),
                'min': round(float(df[col].min()), 2),
                'max': round(float(df[col].max()), 2)
            })
    return results


def detect_outliers_zscore(df, threshold=3.0):
    results = []
    for col in df.select_dtypes(include='number').columns:
        mean, std = df[col].mean(), df[col].std()
        if std == 0:
            continue
        z = (df[col] - mean) / std
        n_out = (z.abs() > threshold).sum()
        if n_out > 0:
            results.append({'column': col, 'method': 'Z-score',
                            'n_outliers': int(n_out),
                            'pct': round(n_out / len(df) * 100, 2)})
    return results

print('Funzioni pronte.')

In [ ]:
results = {}

for name, df in datasets.items():
    print(f'Analisi: {name}...')

    # --- Missing values ---
    miss = classify_missingness(df)

    # --- Outliers ---
    out_iqr = detect_outliers_iqr(df)
    out_z   = detect_outliers_zscore(df)

    # --- LLM: missing ---
    if miss:
        miss_prompt = (
            f'Tabella: {name} ({len(df):,} righe)\n'
            f'Valori mancanti:\n{json.dumps(miss, indent=2, ensure_ascii=False)}\n\n'
            f'Per ogni colonna: classifica MCAR/MAR/MNAR con motivazione, '
            f'impatto sul DW, strategia raccomandata.'
        )
        llm_missing = call_llm(SYSTEM_MISSING, miss_prompt)
    else:
        llm_missing = 'Nessun valore mancante rilevato in questa tabella.'

    # --- LLM: outliers ---
    if out_iqr:
        out_prompt = (
            f'Tabella: {name} ({len(df):,} righe)\n'
            f'Outlier rilevati (IQR):\n{json.dumps(out_iqr, indent=2, ensure_ascii=False)}\n\n'
            f'Per ogni colonna con outlier: cause plausibili nel contesto Olist, '
            f'impatto sul DW, strategia raccomandata.'
        )
        llm_outliers = call_llm(SYSTEM_OUTLIER, out_prompt)
    else:
        llm_outliers = 'Nessun outlier significativo rilevato in questa tabella.'

    results[name] = {
        'missing': miss, 'outliers_iqr': out_iqr, 'outliers_z': out_z,
        'llm_missing': llm_missing, 'llm_outliers': llm_outliers
    }

    # --- Display inline ---
    display(Markdown(f'---\n### {name.upper()}'))
    display(Markdown(f'**Valori mancanti ({len(miss)} colonne)**\n\n' + llm_missing))
    display(Markdown(f'**Outlier IQR ({len(out_iqr)} colonne)**\n\n' + llm_outliers))

    # --- Salva file ---
    out_file = OUT_PATH + f'L2_{name}_analysis.md'
    with open(out_file, 'w', encoding='utf-8') as f:
        f.write(f'# L2 - Missing & Outlier Analysis: {name}\n\n')
        f.write(f'**Data:** {datetime.now().strftime("%Y-%m-%d %H:%M")} | **Provider:** {LLM_PROVIDER}\n\n')
        f.write('## Valori Mancanti\n\n')
        f.write('```json\n' + json.dumps(miss, indent=2, ensure_ascii=False) + '\n```\n\n')
        f.write('### Interpretazione LLM\n\n' + llm_missing + '\n\n')
        f.write('## Outlier\n\n')
        f.write('```json\n' + json.dumps(out_iqr, indent=2, ensure_ascii=False) + '\n```\n\n')
        f.write('### Narrativa LLM\n\n' + llm_outliers + '\n')
    print(f'  Salvato -> {out_file}')

print(f'\nAnalisi completata per {len(results)} tabelle.')

In [ ]:
# Riepilogo aggregato
agg_file = OUT_PATH + 'L2_RIEPILOGO.md'
with open(agg_file, 'w', encoding='utf-8') as f:
    f.write('# L2 - Riepilogo Missing & Outlier\n\n')
    f.write('| Tabella | Righe | Col. con null | Col. con outlier IQR |\n')
    f.write('|---------|-------|---------------|----------------------|\n')
    for name, res in results.items():
        df = datasets[name]
        f.write(f'| {name} | {len(df):,} | {len(res["missing"])} | {len(res["outliers_iqr"])} |\n')
print(f'Riepilogo salvato: {agg_file}')